In [1]:
import sys

if "google.colab" in sys.modules:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    original_data = '/content/drive/My Drive/MS1/original_dataset'
    final_data = '/content/drive/My Drive/MS1/Final_Dataset'

    # Install required packages
    !pip install pymatgen torch_geometric mp_api

else:
    original_data = "original_dataset"
    final_data = "Final_Dataset"

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, NNConv, CGConv, global_max_pool, GCNConv

from pymatgen.core import Structure, PeriodicSite, DummySpecie, Composition, Element
from pymatgen.core.periodic_table import Element as PMGElement
from pymatgen.analysis.local_env import MinimumDistanceNN, CrystalNN, VoronoiNN
# from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# import json
# import config
# import graphy_gnn
import graphy_cvae

/home/amutua/inverse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Prep

In [3]:
# The whole dataset
comb_df = pd.read_csv(f"{final_data}/combined/combined_data.csv")

# Sample row
sample_row = comb_df.iloc[0]

# Get defective structure
material = sample_row["dataset_material"]
id = sample_row["_id"]

# Get defective structure
defective_file_path = f"{original_data}/{material}/cifs/{id}.cif"
defective_structure = Structure.from_file(defective_file_path)

# Get reference structure
ref_file_path = f"{final_data}/ref_cifs/{material}.cif"
reference_structure = Structure.from_file(ref_file_path)

/home/amutua/inverse/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 32 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


In [10]:
gnn_train_graphs = torch.load(f"{final_data}/train_graphs.pt", weights_only=False)
gnn_val_graphs = torch.load(f"{final_data}/val_graphs.pt", weights_only=False)
gnn_test_graphs = torch.load(f"{final_data}/test_graphs.pt", weights_only=False)

# Find the graph where the edge features has shape 0 and remove them

def remove_1_defect_graphs(graphs):
    unwanted = []
    for i, data in enumerate(graphs):
        if data.edge_attr.shape[0] == 0:
            unwanted.append(i)

    new_graphs = [data for i, data in enumerate(graphs) if i not in unwanted]
    return new_graphs

new_train_graphs = remove_1_defect_graphs(gnn_train_graphs)
new_val_graphs = remove_1_defect_graphs(gnn_val_graphs)
new_test_graphs = remove_1_defect_graphs(gnn_test_graphs)

In [76]:
def collate_fn(batch):
    batch_node_features = []
    batch_edge_index = []
    batch_defect_cloud = []
    batch_band_gap = []

    node_offset = 0
    for item in batch:
        batch_node_features.append(item["node_features"])
        batch_defect_cloud.append(item["defect_cloud"])
        batch_band_gap.append(item["band_gap"])

        edge_index = item["edge_index"] + node_offset
        batch_edge_index.append(edge_index)

        node_offset += item["node_features"].shape[0]

    return {
        "node_features": torch.cat(batch_node_features, dim=0),
        "edge_index": torch.cat(batch_edge_index, dim=1),
        "defect_cloud": torch.stack(batch_defect_cloud, dim=0),
        "band_gap": torch.stack(batch_band_gap, dim=0)
    }

In [12]:
train_loader = DataLoader(new_train_graphs, batch_size=1, shuffle=True) # , collate_fn=collate_fn)

for batch in train_loader:
    sample_batch = batch
    break

In [14]:
NODE_DIM = sample_batch.x.shape[1] 
# EDGE_DIM = sample_batch.edge_attr.shape[1]

HIDDEN_DIM = 128
LATENT_DIM = 32
MAX_POINTS = int(max(comb_df["defect_sites"]))

## The model architecture

### Model 1

In [15]:
#  ENCODER
class PristineGNNEncoder(nn.Module):
    def __init__(self, node_dim, hidden_dim, latent_dim):
        super().__init__()

        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # cloud_flat = n_max * point_dim
        
        self.fc_combine = nn.Linear(hidden_dim * 2, hidden_dim)

        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x_mean = global_mean_pool(x, batch)
        x_max  = global_max_pool(x, batch)


        x_graph = torch.cat([x_mean, x_max], dim=-1)

        # B = h_graph.shape[0]
        # cloud_flat = defect_cloud.view(B, -1)  # (B, n_max*point_dim)

        # combined = torch.cat([h_graph, cloud_flat], dim=-1)
        
        hidden   = F.relu(self.fc_combine(x_graph))

        mu      = self.fc_mu(hidden)       # (B, latent_dim)
        log_var = self.fc_log_var(hidden)  # (B, latent_dim)
        return mu, log_var

In [16]:
test_encoder = PristineGNNEncoder(NODE_DIM, HIDDEN_DIM, LATENT_DIM)
mu, log_var = test_encoder(sample_batch)

In [18]:
# Reparametize to decode
def reparameterize(mu, log_var):
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)
    return mu + eps * std

latent_vector = reparameterize(mu, log_var)
print("Latent vector shape:", latent_vector.shape)

Latent vector shape: torch.Size([1, 32])


In [19]:
# DECODER
class DefectCloudDecoder1(nn.Module):
    def __init__(self,hidden_dim, latent_dim, n_max, node_dim):
        super().__init__()
        self.n_max     = n_max
        self.node_dim = node_dim

        # +1 for band gap conditioning
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, n_max * node_dim),
        )

    def forward(self, z, band_gap):

        if band_gap.dim() == 1:
            band_gap = band_gap.unsqueeze(-1)

        inp = torch.cat([z, band_gap], dim=-1)

        out = self.net(inp)
        out = out.view(-1, self.n_max, self.node_dim)
        return out

        """# Apply activations per channel
        coords  = torch.sigmoid(out[..., :3])
        z_vals  = torch.sigmoid(out[..., 3:5])
        logits  = out[..., 5:]

        return torch.cat([coords, z_vals, logits], dim=-1)"""

In [20]:
test_decoder = DefectCloudDecoder1(HIDDEN_DIM, LATENT_DIM, MAX_POINTS, NODE_DIM)

recon_decoded = test_decoder(latent_vector, sample_batch.y)

In [21]:
recon_decoded

tensor([[[-1.5159e-01, -2.9819e-02,  2.1407e-02,  3.0248e-01, -5.1360e-02,
          -5.6118e-02,  2.3766e-01,  2.1170e-02, -1.1079e-01,  1.3352e-01,
          -2.1653e-01,  2.4424e-01,  1.5766e-01,  8.7129e-03,  5.0249e-02,
          -4.2674e-02, -1.6154e-01, -2.7831e-01, -3.3310e-02,  7.2307e-03,
           2.0791e-01, -1.0447e-01,  3.7563e-01, -7.9550e-02, -4.8036e-02,
           2.5156e-01, -6.6609e-02],
         [-3.2878e-02,  4.1582e-02, -8.4982e-02,  5.8316e-02, -2.3760e-02,
           2.9630e-02, -2.3247e-02, -2.2550e-01, -1.2759e-02,  2.4136e-01,
           5.8476e-02,  7.7037e-02,  1.6548e-01,  1.0283e-01, -2.8201e-02,
           7.8449e-02, -4.6148e-02,  1.5330e-01, -7.3864e-04,  6.0904e-02,
           4.9425e-03,  3.8519e-02, -1.5065e-01, -2.1918e-01,  9.1849e-02,
           1.8337e-01, -2.0760e-01],
         [-7.4687e-02, -1.4519e-01, -8.7891e-02,  6.4724e-02,  2.1892e-01,
           3.7538e-01, -6.5506e-02, -8.5249e-04,  7.8209e-03,  7.0909e-02,
           2.6660e-01,  6.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim = NODE_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM, n_max = MAX_POINTS, point_dim = POINT_DIM):

        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = PristineGNNEncoder(node_dim, hidden_dim, latent_dim, n_max, point_dim)
        self.decoder = DefectCloudDecoder(hidden_dim,latent_dim, n_max, point_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data.node_features, data.edge_index, data.defect_cloud, data.batch)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.band_gap)
        return recon, mu, log_var

    @torch.no_grad()
    def generate(self, target_band_gap, n_samples, device = "cpu"):
        self.eval()
        z = torch.randn(n_samples, self.latent_dim, device=device)
        bg = torch.full((n_samples,), target_band_gap,
                        dtype=torch.float32, device=device)
        cloud_tensors = self.decoder(z, bg)

        results = []
        for i in range(n_samples):
            points = graphy_cvae.tensor_to_cloud(cloud_tensors[i], threshold=0.1)
            results.append(points)
        return results

### Model 2

In [ ]:
#  ENCODER
class PristineGNNEncoder(nn.Module):
    def __init__(self, node_dim = NODE_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM, n_max = MAX_POINTS, point_dim = POINT_DIM):
        super().__init__()

        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        cloud_flat = n_max * point_dim
        
        self.fc_combine = nn.Linear(hidden_dim * 2 + cloud_flat, hidden_dim)

        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    # def forward(self, data):
    def forward(self,
            node_features: torch.Tensor,  # (total_nodes, node_dim)
            edge_index:    torch.Tensor,  # (2, total_edges)
            defect_cloud:  torch.Tensor,  # (B * n_max, point_dim)
            batch:         torch.Tensor   # (total_nodes,) – node→graph mapping
            ) -> tuple[torch.Tensor, torch.Tensor]:
        '''node_features = data.node_features
        edge_index = data.edge_index
        defect_cloud = data.defectt_cloud
        batch = data.batch '''

        h = F.relu(self.conv1(node_features, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        h_mean = global_mean_pool(h, batch)
        h_max  = global_max_pool(h, batch)
        h_graph = torch.cat([h_mean, h_max], dim=-1)

        B = h_graph.shape[0]
        cloud_flat = defect_cloud.view(B, -1)  # (B, n_max*point_dim)

        combined = torch.cat([h_graph, cloud_flat], dim=-1)
        hidden   = F.relu(self.fc_combine(combined))

        mu      = self.fc_mu(hidden)       # (B, latent_dim)
        log_var = self.fc_log_var(hidden)  # (B, latent_dim)
        return mu, log_var

In [ ]:
# Reparametize to decode
def reparameterize(mu, log_var):
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)
    return mu + eps * std

latent_vector = reparameterize(mu, log_var)
print("Latent vector shape:", latent_vector.shape)  # Should be (B, latent

In [102]:
ALL_ELEMENTS = [el.symbol for el in Element]   # length 118
N_SPECIES    = len(ALL_ELEMENTS)               # 118

def build_species_mask(allowed_symbols):
    """
    Returns a boolean mask of shape (N_SPECIES,).
    True  → species is allowed (logit kept).
    False → species is forbidden (logit set to -inf).
    """
    allowed_set = set(allowed_symbols)
    mask = torch.tensor(
        [el in allowed_set for el in ALL_ELEMENTS],
        dtype=torch.bool
    )
    return mask

In [108]:
ALLOWED_ELEMENTS = ["Mo", "S", "W", "Se", "B", "N", "Ga", "In", "P", "V", "O", "C"]
ALLOWED_MASK = build_species_mask(ALLOWED_ELEMENTS)

N_DEFECT_TYPES = 2 # vacancy and substitution

class DefectCloudDecoder(nn.Module):
    def __init__(
            self, 
            hidden_dim = HIDDEN_DIM, 
            latent_dim = LATENT_DIM, 
            n_max = MAX_POINTS, 
            allowed_species = ALLOWED_ELEMENTS, 
            allowed_mask = ALLOWED_MASK
        ):
        super().__init__()

        self.n_max = n_max
        self.n_defect_types = N_DEFECT_TYPES          # 2: vacancy, substitution
        self.n_species = N_SPECIES               # 118
        self.species_mask = allowed_mask  # boolean mask for allowed species
        self.defect_mask = torch.ones(self.n_defect_types, dtype=torch.bool)  # all defect types allowed by default

        # point_dim = 3 (coords) + 2 (z-vals) + N_DEFECT_TYPES + N_SPECIES
        self.point_dim = NODE_DIM + self.n_defect_types + self.n_species

        
        out_dim = n_max * self.point_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, out_dim),
        )



    def forward(self, z: torch.Tensor, band_gap: torch.Tensor) -> torch.Tensor:
        if band_gap.dim() == 1:
            band_gap = band_gap.unsqueeze(-1)

        B   = z.shape[0]
        inp = torch.cat([z, band_gap], dim=-1)
        out = self.net(inp).view(B, self.n_max, self.point_dim)

        # Split channels
        coords         = torch.sigmoid(out[..., :3])
        z_vals         = torch.sigmoid(out[..., 3:5])
        defect_logits  = out[..., 5 : 5 + self.n_defect_types]
        species_logits = out[..., 5 + self.n_defect_types :]

        # Apply masks → forbidden logits become -inf before any softmax/sampling
        # defect_logits, species_logits = self._apply_masks(defect_logits, species_logits)

        return torch.cat([coords, z_vals, defect_logits, species_logits], dim=-1)

In [109]:
# Test 1st decoder
the_decoder = DefectCloudDecoder(HIDDEN_DIM, LATENT_DIM, MAX_POINTS, POINT_DIM) # .to(device)
reconstructed_cloud = the_decoder(latent_vector, sample_bg)
print("Reconstructed cloud shape:", reconstructed_cloud.shape)  # Should be (B, n_max, point_dim)

Reconstructed cloud shape: torch.Size([1, 24, 125])


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim = NODE_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM, n_max = MAX_POINTS, point_dim = POINT_DIM):

        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = PristineGNNEncoder(node_dim, hidden_dim, latent_dim, n_max, point_dim)
        self.decoder = DefectCloudDecoder(hidden_dim,latent_dim, n_max, point_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data.node_features, data.edge_index, data.defect_cloud, data.batch)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.band_gap)
        return recon, mu, log_var

    @torch.no_grad()
    def generate(self, target_band_gap, n_samples, device = "cpu"):
        self.eval()
        z = torch.randn(n_samples, self.latent_dim, device=device)
        bg = torch.full((n_samples,), target_band_gap,
                        dtype=torch.float32, device=device)
        cloud_tensors = self.decoder(z, bg)

        results = []
        for i in range(n_samples):
            points = graphy_cvae.tensor_to_cloud(cloud_tensors[i], threshold=0.1)
            results.append(points)
        return results

### Loss Function

In [36]:
# Loss function for cvae
def cvae_loss(recon, target, mu, log_var, beta): # =1e-3):
    # coords
    recon_coords = recon[..., :3]
    target_coords = target[..., :3]
    coords_loss = F.mse_loss(recon_coords, target_coords)

    # Props
    recon_props = recon[..., 3:17]
    target_props = target[..., 3:17]
    props_loss = F.mse_loss(recon_props, target_props)

    # sites
    recon_sites = recon[..., 17:]
    target_sites = recon[..., 17:]
    sites_loss = F.cross_entropy(recon_sites, target_sites)


    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())

    total = coords_loss + props_loss + sites_loss + beta * kl_loss
    return total, coords_loss, props_loss, sites_loss, kl_loss

cvae_loss(recon, the_sample.x, mu, log_var, 1e-3)

(tensor(4.0770, device='cuda:0', grad_fn=<AddBackward0>),
 tensor(0.2903, device='cuda:0', grad_fn=<MseLossBackward0>),
 tensor(3.7896, device='cuda:0', grad_fn=<MseLossBackward0>),
 tensor(-0.0034, device='cuda:0', grad_fn=<DivBackward1>),
 tensor(0.5890, device='cuda:0', grad_fn=<MulBackward0>))

## Training and Testing

In [ ]:
# Loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# Parameters for training model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DefectCVAE(NODE_DIM, EDGE_DIM, U_DIM, HIDDEN_DIM, LATENT_DIM).to(device)
EPOCHS = 40
beta = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.001)
losses_dict = {"train_loss": [], "val_loss": [], "kl": [], "mse": []}

# Validate function
def validate(model, loader, beta):
    # Model in evaluation mode
    model.eval()
    t_loss, tc_loss, tp_loss, ts_loss, kl = 0.0, 0.0, 0.0, 0.0, 0.0

    with torch.no_grad():
        for batch in loader:
            # batch = {k: v.to(device) for k, v in batch.items()}
            recon, mu, log_var = model(batch.to(device))
            recon = recon.view((144,20))

            t, c_loss, p_loss, s_loss, kl_loss = cvae_loss(recon, batch.x, mu, log_var, beta)

            t_loss +=t.item() * batch.num_graphs
            tc_loss += c_loss.item() * batch.num_graphs
            tp_loss += p_loss.item() * batch.num_graphs
            ts_loss += s_loss.item() * batch.num_graphs
            kl += kl_loss.item() * batch.num_graphs

        n = len(loader.dataset)
        totals = {"the_loss": t_loss/n, "coordinates": tc_loss/n, "props": tp_loss/n, "sites": ts_loss/n ,"kl": kl/n}
        return totals

# Training loop
for epoch in range(1, EPOCHS+1):
    # Put model in training mode at the start of each epoch
    model.train()
    t_loss, tc_loss, tp_loss, ts_loss, kl = 0.0, 0.0, 0.0, 0.0, 0.0
    current_beta = beta * min(1.0, epoch / EPOCHS)

    for batch in train_loader:
        optimizer.zero_grad()

        # Output of the model
        recon, mu, log_var = model(batch.to(device))
        recon = recon.view((144, 20))

        
        t, c_loss, p_loss, s_loss, kl_loss = cvae_loss(recon, batch.x, mu, log_var, current_beta)

        t.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        t_loss +=t.item() * batch.num_graphs
        tc_loss += c_loss.item() * batch.num_graphs
        tp_loss += p_loss.item() * batch.num_graphs
        ts_loss += s_loss.item() * batch.num_graphs
        kl += kl_loss.item() * batch.num_graphs

    n = len(train_loader)
    # t_losses = {"the_loss": t_loss/n, "mse": t_mse/n, "kl": t_kl/n}
    t_losses = {"the_loss": t_loss/n, "coordinates": tc_loss/n, "props": tp_loss/n, "sites": ts_loss/n ,"kl": kl/n}

    # Validate at the end of each epoch
    val_losses = validate(model, val_loader, current_beta)
    scheduler.step()

    print(f"Epoch {epoch:4d}\ntrain: {t_losses['the_loss']:.4f} | {t_losses['coordinates']:.4f} | {t_losses['props']:.4f} | {t_losses['sites']:.4f} | {t_losses['kl']:.4f}\nval: {val_losses['the_loss']:.4f} | {val_losses['coordinates']:.4f} | {val_losses['props']:.4f} | {val_losses['sites']:.4f} | {val_losses['kl']:.4f}\n")

/home/amutua/inverse/lib/python3.10/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch    1
train: -76499157901555173163008.0000 | 31682117782172557312.0000 | 276250818272827146240.0000 | -76807625997551749038080.0000 | 21436524718349894025216.0000
val: 556698013925048832.0000 | 93063.7914 | 383449.1788 | -176724567.9360 | 22267921201846007365632.0000

Epoch    2
train: -62260969498287456387072.0000 | 14032269360895795200.0000 | 159918184835010002944.0000 | -62436042538310009094144.0000 | 22459866629760023003136.0000
val: 1115382586512399872.0000 | 1051229.8990 | 5274457.8840 | -2381886204.9280 | 22307652374271264555008.0000

Epoch    3
train: -65818910950417630232576.0000 | 13046430828422455296.0000 | 158898668232816820224.0000 | -65992543279697872551936.0000 | 22493798347403356536832.0000
val: 1674753846292171008.0000 | 5993031.5680 | 28903891.9520 | -15292949356.5440 | 22330050213907156434944.0000

Epoch    4
train: -59082045126976349405184.0000 | 8440679942335298560.0000 | 97066895192869273600.0000 | -59189803858828147556352.0000 | 22517132481109279375360.0000


KeyboardInterrupt: 

## Inverse Design

In [50]:
def inverse_design(model, pristine, target_band_gap, device):
    model.eval()
    model = model.to(device)

    # Generate defective structure
    gen_def_structure = model.generate(pristine, target_band_gap)

   
    return gen_def_structure

origin = ref_sample = Structure.from_file(f"{final_data}/ref_cifs/high_GaSe.cif")
first_struct = inverse_design(model, origin, 1.3, "cpu")

In [51]:
print(first_struct)

Full Formula (Rb2 Mn1 V1 Zn14 Cr1 Ga15 Fe2 Co9 Cu15 Ni8 Ge26 As15 Se10 Br16 Kr9)
Reduced Formula: Rb2MnVZn14CrGa15Fe2Co9Cu15Ni8Ge26As15Se10Br16Kr9
abc   :  22.914400  22.914400  20.000000
angles:  90.000000  90.000000 120.000012
pbc   :       True       True       True
Sites (144)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  Ni    0.112623  0.056048  0.377355
  1  Ge    0.108528  0.219112  0.379291
  2  Ni    0.108108  0.38448   0.378505
  3  Ge    0.109691  0.551017  0.37814
  4  Co    0.110627  0.714196  0.378738
  5  Ge    0.107597  0.870086  0.378252
  6  Cu    0.274656  0.055681  0.379102
  7  Cr    0.276782  0.221911  0.380982
  8  Co    0.274569  0.383582  0.38129
  9  As    0.273537  0.548997  0.378453
 10  Ge    0.27584   0.718054  0.379689
 11  Cu    0.276124  0.882788  0.381675
 12  Cu    0.439068  0.057326  0.378207
 13  As    0.441728  0.222276  0.379616
 14  Fe    0.437897  0.38488   0.378116
 15  Ge    0.43846   0.54949   0.387496
